In [1]:
%pip install requests pandas openpyxl argparse
import getpass
import argparse
import json
import os
import re
import threading
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
import pandas as pd
import requests

  Using cached argparse-1.4.0-py2.py3-none-any.whl.metadata (2.8 kB)
Using cached argparse-1.4.0-py2.py3-none-any.whl (23 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


# STEP 1: Settings

In [2]:
HOST  = "https://maki.uni-mannheim.de/v1"
MODEL = "gemma4-26b"
TEMPERATURE = 0           # 0 = deterministic / predictable
SEED = 123
MAX_WORKERS = 2              # Parallel calls; lower it if the server returns errors
TIMEOUT = 180                # Seconds to wait for one answer
MAX_RETRIES = 3              # Tries per paragraph before it is marked as an error
STOP_AFTER_ERRORS = 20       # Pause the whole run if this many fail in a row

# Prompt for API Key
API_KEY = os.environ.get("UNI_API_KEY") or getpass.getpass("Paste the Uni API key: ")
print("Key loaded:", bool(API_KEY))

Key loaded: True


# STEP 2: The Annotation Prompt

In [3]:
if "API_KEY" not in globals():
    raise RuntimeError("Please run STEP 1 first.")

# Customize your labels and definition here
ANNOTATION_PROMPT = """

ROLE
You are a text annotator specializing in media content analysis and computational communication. Your task is to apply the Stereotype Content Model (SCM) to evaluate whether targeted social groups in translated German newspaper articles are portrayed with Warmth.

TASK
Based on the multiple paragraphs provided, decide whether [Gruppe 1] (or an individual member of [Gruppe 1]) is attributed with warmth. The label is TRUE if the text contains warmth and FALSE if it does not. Prioritize explicit mentions of warm attributes. Implicit portrayals are also encouraged to be coded when the text clearly conveys warmth without using a trait word. Give a short explanation of why you chose TRUE or FALSE, based on the criteria below. While coding, target only [Gruppe 1]. Do NOT code whether other groups ([andere Gruppe]) display warmth toward [Gruppe 1].

CONTEXT
You are working with paragraphs from German newspaper articles from a wide range of the political spectrum. The paragraphs have been translated into English. Each input consists of several consecutive paragraphs from one article, together with the target group [Gruppe 1]. Return one label for the whole passage. Entity names in the text are masked as [Gruppe 1] and [andere Gruppe]. Keep these names exactly as written and do not translate, change or replace them, including in your explanation.

DEFINITION
Warmth in news coverage is defined as an individual's or group's perceived positive intention towards the perceiver or their group of belonging. Warmth is conceptualized through two dimensions in media reporting:

Friendliness/sociability: Framing the individual or group as friendly if they are depicted as someone who treats others in a friendly, kind and helpful manner.
Attribute markers include but are not limited to: sociable, friendly, warm, liked, outgoing, likeable, pleasant, popular

Morality: Framing the individual or group as moral if they are attributed as someone trustworthy, honest, fair and good.
Attribute markers include but are not limited to: moral, trustworthy, sincere, fair, tolerant, altruistic, benevolent

CRITERIA: WARMTH OCCURS IF

Case: A warm trait is attributed to [Gruppe 1] or a member.
Example: "The [Gruppe 1] are considered welcoming / hospitable."

Case: A synonym or noun form of a warm trait is used.
Example: "Their honesty is praised."

Case: A behaviour that clearly shows good intentions toward others.
Example: "The [Gruppe 1] native collected donations for the refugees."

Case: A positive moral evaluation.
Example: "The [Gruppe 1] fight heroically."

CRITERIA: WARMTH DOES NOT OCCUR IF

Case: Others are warm TOWARD the group.
Example: "The local population showed great readiness to help the [Gruppe 1]." / "The public showed great willingness to assist [Gruppe 1]."
Why not: The warmth belongs to the Germans, not to [Gruppe 1].

Case: The group is only a victim.
Example: "Three [Gruppe 1] were attacked."
Why not: Being a victim is not a trait.

Case: It is about ability or courage (competence).
Example: "The brave [Gruppe 1]", "hard-working"
Why not: Courage, diligence and skill = competence.

Case: The warm word describes someone else.
Example: "Filip takes care of everyone," says the [Gruppe 1].
Why not: [Gruppe 1] is only the speaker.

Case: It is only a benefit frame.
Example: "[Gruppe 1] strengthen the economy."
Why not: An economic benefit, not a trait (Frames dimension).

Case: No portrayal at all.
Example: "In 2022, 12,000 [Gruppe 1] applied for asylum."
Why not: Facts and statistics only.

DECISION RULES
tba.

OUTPUT FORMAT
Return only a valid JSON object, with no text before or after it. The object has exactly two keys, in this order:
"explanation": 1-2 sentences explaining the label, citing the specific words or behavior that led to it.
"label": the string "TRUE" or "FALSE".

Example of valid output:
{"explanation": "The text describes the [Gruppe 1] as hospitable and welcoming toward newcomers.", "label": "TRUE"}

"""

instructions = ANNOTATION_PROMPT
reminder = None  # this is just an additonal vairable to nudge the llm to a specific direction
json_mode= True

print("--- Instructions to annotate WARMTH Loaded ---")
print(instructions)

--- Instructions to annotate WARMTH Loaded ---


ROLE
You are a text annotator specializing in media content analysis and computational communication. Your task is to apply the Stereotype Content Model (SCM) to evaluate whether targeted social groups in translated German newspaper articles are portrayed with Warmth.

TASK
Based on the multiple paragraphs provided, decide whether [Gruppe 1] (or an individual member of [Gruppe 1]) is attributed with warmth. The label is TRUE if the text contains warmth and FALSE if it does not. Prioritize explicit mentions of warm attributes. Implicit portrayals are also encouraged to be coded when the text clearly conveys warmth without using a trait word. Give a short explanation of why you chose TRUE or FALSE, based on the criteria below. While coding, target only [Gruppe 1]. Do NOT code whether other groups ([andere Gruppe]) display warmth toward [Gruppe 1].

CONTEXT
You are working with paragraphs from German newspaper articles from a wide range of 

# STEP 3: Paragraph Input Data

In [4]:
if "API_KEY" not in globals():
    raise RuntimeError("Please run STEP 1 first.")

INPUT_EXCEL_FILE = "sample_5k_translated.xlsx"
CHECKPOINT_FILE = "warmth_annotation_checkpoint.jsonl"
OUTPUT_EXCEL = "sample_5k_warmth_annotated.xlsx"
OUTPUT_JSON = "sample_5k_warmth_annotated.json"

try:
    sample = pd.read_excel(INPUT_EXCEL_FILE)
    if "row_id" not in sample.columns:
        sample.insert(0, "row_id", range(len(sample)))
    print(f"Successfully loaded {len(sample)} paragraph(s) from '{INPUT_EXCEL_FILE}'.")
except FileNotFoundError:
    raise FileNotFoundError(f"Error: Could not find '{INPUT_EXCEL_FILE}' in the current folder.")


Successfully loaded 5000 paragraph(s) from 'sample_5k_translated.xlsx'.


# STEP 4: Set up LLM API Call Function

In [5]:
if "API_KEY" not in globals():
    raise RuntimeError("Please run STEP 1 first.")

def call_llm(text, instructions, reminder=None, max_retries=MAX_RETRIES):
    user_msg = f"Text: '{text}'"
    if reminder:
        user_msg += f"\nReminder: {reminder}"
        
    body = {
        "model": MODEL,
        "messages": [
            {"role": "system", "content": instructions},
            {"role": "user",   "content": user_msg},
        ],
        "temperature": TEMPERATURE,
        "seed": SEED,
        "reasoning_effort": "none",
        "response_format": {"type": "json_object"},
        "max_tokens": 1000,
    }
    
    last_error = None
    if json_mode:
        body["response_format"] = {"type": "json_object"}

        
    for attempt in range(1, max_retries + 1):
        try:
            r = requests.post(f"{HOST}/chat/completions",
                              headers={"Authorization": f"Bearer {API_KEY}"},
                              json=body, timeout=TIMEOUT)
                                             
            r.raise_for_status()
            j = r.json()
            return {
                "output": j["choices"][0]["message"]["content"].strip(),
                "tokens_prompt": j["usage"]["prompt_tokens"],
                "tokens_response": j["usage"]["completion_tokens"],
                "error": None,
            }
            
        except requests.exceptions.HTTPError as he:
            last_error = str(he)
            if he.response.status_code == 504:
                # 504 Gateway Timeout: back off aggressively 
                wait_time = 15 * (2 ** (attempt - 1))
                print(f"Attempt {attempt}/{max_retries} failed: 504 Gateway Timeout. Server is overloaded. Waiting {wait_time}s...")
                time.sleep(wait_time)
            else:
                print(f"Attempt {attempt}/{max_retries} failed with HTTP error: {he}")
                time.sleep(5 * attempt)
                # Back off softly if too many requests are sent

            if r.status_code == 429:
                                wait_time = 60
                                print(f"Attempt {attempt}/{max_retries} failed: 429 Too Many Requests. Cooling down for {wait_time}s...")
                                time.sleep(wait_time)
                                continue

            if he.response.status_code is not None and 400 <= he.response.status_code < 500 and he.response.status_code not in (408, 429):
                break

        except Exception as e:
            last_error = str(e)
            print(f"Attempt {attempt}/{max_retries} failed (Network error): {e}")
            time.sleep(5 * attempt)
            
    print("All retry attempts failed for this entry.")
    return {"output": None, "tokens_prompt": None, "tokens_response": None, "error": last_error}

# STEP 5: Run Annotation Loop with parallelization 

In [6]:
if "API_KEY" not in globals() or "sample" not in globals():
    raise RuntimeError("Please run STEP 1 and STEP 3 first.")

print(f"Starting annotation loop for {len(sample)} entries...")

# 1. Load existing checkpoints if any (enables resuming if interrupted)
completed_ids = set()
results_cache = {}

if os.path.exists(CHECKPOINT_FILE):
    with open(CHECKPOINT_FILE, "r", encoding="utf-8") as f:
        for line in f:
            try:
                data = json.loads(line)
                row_id = data.get("row_id")
                if row_id is not None:
                    completed_ids.add(row_id)
                    results_cache[row_id] = data
            except json.JSONDecodeError:
                continue
    print(f"Resumed from checkpoint: {len(completed_ids)} already annotated entries found.")

# Thread-safe lock for writing to the checkpoint file
file_lock = threading.Lock()

def process_row(row):
    row_id = row["row_id"]
    if row_id in completed_ids:
        return results_cache[row_id]
    
    # Explicitly grab text from the 'translation' column
    text_content = str(row.get("translation", ""))
    
    response = call_llm(text_content, instructions, reminder=reminder)
    
    # Parse the LLM output JSON string
    parsed_output = {"label": None, "explanation": "Failed to parse JSON"}
    
    if response["output"]:
        try:
            clean_out = response["output"].strip()
            if clean_out.startswith("```json"):
                clean_out = clean_out[7:]
            if clean_out.startswith("```"):
                clean_out = clean_out[3:]
            if clean_out.endswith("```"):
                clean_out = clean_out[:-3]
            
            parsed_json = json.loads(clean_out.strip())
            parsed_output["label"] = parsed_json.get("label")
            parsed_output["explanation"] = parsed_json.get("explanation")
        except Exception as e:
            parsed_output["explanation"] = f"JSON parse error: {e} | Raw: {response['output']}"

    result_entry = {
        "row_id": row_id,
        "label": parsed_output["label"],
        "explanation": parsed_output["explanation"],
        "raw_output": response["output"],
        "error": response["error"],
        "tokens_prompt": response["tokens_prompt"],
        "tokens_response": response["tokens_response"]
    }
    
    with file_lock:
        with open(CHECKPOINT_FILE, "a", encoding="utf-8") as cp:
            cp.write(json.dumps(result_entry, ensure_ascii=False) + "\n")
            
    return result_entry

# 2. Run execution using ThreadPoolExecutor
results = []
errors_in_a_row = 0

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {executor.submit(process_row, row): row for _, row in sample.iterrows()}
    
    completed_count = len(completed_ids)
    total_count = len(sample)
    
    for future in as_completed(futures):
        res = future.result()
        results.append(res)
        completed_count += 1
        
        if res.get("error"):
            errors_in_a_row += 1
        else:
            errors_in_a_row = 0
            
        if errors_in_a_row >= STOP_AFTER_ERRORS:
            print(f"Stopping early: {STOP_AFTER_ERRORS} errors encountered in a row.")
            break
            
        if completed_count % 10 == 0 or completed_count == total_count:
            print(f"Progress: {completed_count}/{total_count} processed.")

# 3. Merge results back into DataFrame and save final outputs
results_df = pd.DataFrame(results)
final_df = pd.merge(sample, results_df, on="row_id", how="left")

final_df.to_excel(OUTPUT_EXCEL, index=False)
final_df.to_json(OUTPUT_JSON, orient="records", indent=2, force_ascii=False)

print("\n--- Annotation Complete! ---")
print(f"Checkpoint saved to: {CHECKPOINT_FILE}")
print(f"Excel output saved to: {OUTPUT_EXCEL}")
print(f"JSON output saved to: {OUTPUT_JSON}")

Starting annotation loop for 5000 entries...
Resumed from checkpoint: 3446 already annotated entries found.
Progress: 3450/5000 processed.
Progress: 3460/5000 processed.
Progress: 3470/5000 processed.
Progress: 3480/5000 processed.
Progress: 3490/5000 processed.
Progress: 3500/5000 processed.
Progress: 3510/5000 processed.
Progress: 3520/5000 processed.
Progress: 3530/5000 processed.
Progress: 3540/5000 processed.
Progress: 3550/5000 processed.
Progress: 3560/5000 processed.
Progress: 3570/5000 processed.
Progress: 3580/5000 processed.
Progress: 3590/5000 processed.
Progress: 3600/5000 processed.
Progress: 3610/5000 processed.
Progress: 3620/5000 processed.
Progress: 3630/5000 processed.
Progress: 3640/5000 processed.
Progress: 3650/5000 processed.
Progress: 3660/5000 processed.
Progress: 3670/5000 processed.
Progress: 3680/5000 processed.
Progress: 3690/5000 processed.
Progress: 3700/5000 processed.
Progress: 3710/5000 processed.
Progress: 3720/5000 processed.
Progress: 3730/5000 proc